# Historical Data Binary Outputs

Notebook dedicado ao experimento bin?rio `Direction_t+n` com teste fixo e armazenamento de previs?es/probabilidades.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pickle

from src.data.historical_loader import load_historical_asset_dataframe
from src.data.historical_preprocessing import prepare_historical_market_dataframe
from src.features.historical_features import DEFAULT_HISTORICAL_FEATURES, add_log_diff_targets, build_log_diff_dataset, create_lagged_features
from src.models.historical_training import build_xgb_lgbm_models, get_fixed_test_set, tune_lags_with_fixed_test


In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'CSvs'
ASSET_NAME = 'Binance_BTCUSDT_d'
TARGET_HORIZONS = (1, 3, 7, 14, 30)
MAX_LAGS = 20
FEATURE_COLUMNS = [
    'log_diff_open',
    'log_diff_high',
    'log_diff_low',
    'log_diff_close',
    'log_diff_volume_btc',
    'log_diff_volume_usdt',
    'log_diff_tradecount',
]
MODELS = build_xgb_lgbm_models()


In [ ]:
raw_btc_hist = load_historical_asset_dataframe(DATA_DIR, ASSET_NAME)
btc_hist = prepare_historical_market_dataframe(raw_btc_hist)
btc_hist_transformed = build_log_diff_dataset(btc_hist, feature_columns=DEFAULT_HISTORICAL_FEATURES, keep_close_column=True)
btc_hist_transformed = add_log_diff_targets(btc_hist_transformed, horizons=TARGET_HORIZONS)


In [ ]:
max_horizon = max(TARGET_HORIZONS)
btc_df_common = btc_hist_transformed.iloc[:-max_horizon].copy()
btc_df_common = btc_df_common[(btc_df_common.index >= '2017-11-04') & (btc_df_common.index <= '2024-01-04')]
btc_df_common = create_lagged_features(btc_df_common, FEATURE_COLUMNS, MAX_LAGS).dropna()
test_indices = get_fixed_test_set(btc_df_common)


In [ ]:
results_by_horizon = {}
for horizon in TARGET_HORIZONS:
    target_column = f'Direction_t+{horizon}'
    results_by_horizon[horizon] = {}
    for model_name, model in MODELS.items():
        best_lag, best_accuracy, predictions, probabilities = tune_lags_with_fixed_test(
            btc_df_common,
            FEATURE_COLUMNS,
            target_column,
            MAX_LAGS,
            test_indices,
            n_splits=5,
            model=model,
        )
        results_by_horizon[horizon][model_name] = {
            'best_lags': best_lag,
            'accuracy': best_accuracy,
            'predictions': predictions.tolist() if predictions is not None else [],
            'probabilities': probabilities.tolist() if probabilities is not None else [],
        }
results_by_horizon


In [ ]:
with open(PROJECT_ROOT / 'data' / 'results_by_horizon.pkl', 'wb') as fh:
    pickle.dump(results_by_horizon, fh)
